***Phase-1 : Setup and Data Loading***

In [1]:
pip install langchain sentence-transformers jsonlines

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os

def load_master_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            print(f"✅ Successfully loaded: {file_path}")
            print(f"📊 Total documents found: {len(data)}\n")
            return data
    except FileNotFoundError:
        print(f"❌ Error: The file {file_path} was not found.")
        return []
    except json.JSONDecodeError:
        print(f"❌ Error: The file {file_path} is not valid JSON.")
        return []

govt_file_path = "govt_structured_master.json"
disease_file_path = "who_structured_master_cleaned_safe.json"

print("--- Starting Phase 1: Data Loading ---\n")
govt_data = load_master_json(govt_file_path)
who_data = load_master_json(disease_file_path) # Changed to who_data

if govt_data:
    print(f"Sample Govt Scheme Title: {govt_data[0].get('title', 'No title found')}")
if who_data:
    print(f"Sample Disease Topic Title: {who_data[0].get('title', 'No title found')}")

--- Starting Phase 1: Data Loading ---

✅ Successfully loaded: govt_structured_master.json
📊 Total documents found: 288

✅ Successfully loaded: who_structured_master_cleaned_safe.json
📊 Total documents found: 239

Sample Govt Scheme Title: Janani Shishu Suraksha Karyakram
Sample Disease Topic Title: Abortion


***Phase-2: Chunking & Checkpointing***

In [3]:
pip install langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Updated for BGE-M3 Optimization
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    length_function=len,
    separators=[
        r"\n\n",       
        r"\n",         
        r"(?<=\. )",   
        r" ",          
        r""            
    ],
    is_separator_regex=True
)

def clean_text(raw_text):
    """
    Cleans encoding artifacts, rogue line breaks, and citation numbers.
    """
    # Fix the Rupee symbol encoding error
    text = raw_text.replace("[?]", "₹")
    
    # Remove WHO citation numbers like [1], [22], or (1), (22)
    text = re.sub(r'\s*\[\d+\]\s*', ' ', text)
    text = re.sub(r'\s*\(\d+\)\s*', ' ', text)
    
    # Remove hidden line breaks but keep paragraph breaks
    text = re.sub(r'(?<!\n)\r?\n(?!\n)', ' ', text)
    
    # Remove double spaces
    text = re.sub(r' {2,}', ' ', text)
    
    return text.strip()

def chunk_and_save_data(data_list, output_filepath):
    all_chunks = []
    
    for doc in data_list:
        # 2. Added category and topic for advanced Metadata Filtering
        metadata = {
            "doc_id": doc.get("doc_id"),
            "title": doc.get("title"),
            "category": doc.get("category", "General"),
            "topic": doc.get("topic", "General"),
            "source_name": doc.get("source_name"),
            "source_url": doc.get("source_url")
        }
        
        chunk_counter = 1
        
        for section in doc.get("content", []):
            section_title = section.get("section_title", "General")
            raw_section_text = section.get("text", "")
            
            cleaned_text = clean_text(raw_section_text)
            
            if "faq" in section_title.lower():
                chunk_id = f"{metadata['doc_id']}_chunk_{chunk_counter}"
                chunk_counter += 1
                
                combined_text = f"{section_title}\n{cleaned_text}"
                
                all_chunks.append({
                    "chunk_id": chunk_id,
                    "text": combined_text,
                    "metadata": metadata,
                    "chunk_type": "faq"
                })
                
            else:
                split_texts = text_splitter.split_text(cleaned_text)
                
                for text_chunk in split_texts:
                    chunk_id = f"{metadata['doc_id']}_chunk_{chunk_counter}"
                    chunk_counter += 1
                    
                    final_chunk_text = f"Section: {section_title}\n\n{text_chunk.strip()}"
                    
                    all_chunks.append({
                        "chunk_id": chunk_id,
                        "text": final_chunk_text,
                        "metadata": metadata,
                        "chunk_type": "general"
                    })
                
    with open(output_filepath, 'w', encoding='utf-8') as f:
        for chunk in all_chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
            
    print(f"✅ Processed and saved {len(all_chunks)} chunks to {output_filepath}")
    return all_chunks

print("--- Starting Phase 2: Production Chunking ---")
govt_chunks_raw = chunk_and_save_data(govt_data, "govt_chunks_raw.jsonl")
who_chunks_raw = chunk_and_save_data(who_data, "who_chunks_raw.jsonl")

--- Starting Phase 2: Production Chunking ---
✅ Processed and saved 5090 chunks to govt_chunks_raw.jsonl
✅ Processed and saved 3371 chunks to who_chunks_raw.jsonl
